In [1]:
# import libraries
# import xpress as xp
import math
import numpy as np

In [2]:
col_cnt = 7 #def=7
row_cnt = 9 #def=9
spring_cnt = (col_cnt-1)*row_cnt + (row_cnt-1)*col_cnt #=6 for1row =110 for2D
leg_lb = 2
leg_ub = 6
ntn_half_lb = 0.45
ntn_half_ub = 1.25
lin_str_ub = 22.59
k0_ub = 1000
stiffness_ub = k0_ub * 10

leg_range = [2,4,6]
spring_range = range(spring_cnt) #indexed from zero
legs = 246

In [3]:
p = xp.problem()

In [4]:
leg = [xp.var(name="leg_{}".format(i), lb=leg_lb, ub=leg_ub, vartype=xp.continuous) for i in spring_range]
angle = [xp.var(name="angle_{}".format(i), lb=30, ub=45, vartype=xp.continuous) for i in spring_range]
ntn_half = [xp.var(name="ntn_half_{}".format(i), lb=ntn_half_lb, ub=ntn_half_ub, vartype=xp.continuous) for i in spring_range]
k0 = [xp.var(name="k0{}".format(i), lb=1e-6, ub=k0_ub, vartype=xp.continuous) for i in spring_range] #This is w=0.01 stiffness. orig k0 = [1.76E-5, 4.23E-5]
stiffness = [xp.var(name="stiffness_{}".format(i), lb=1e-6, ub=stiffness_ub, vartype=xp.continuous) for i in spring_range] #This is stiffness including w effect.
str_det = [xp.var(name="str_det_{}".format(i), lb=0.1, ub=lin_str_ub, vartype=xp.continuous) for i in spring_range] #Simulated stretch of detailed serpentine
max_str_det = [xp.var(name="max_str_det_{}".format(i), lb=0.1, ub=lin_str_ub, vartype=xp.continuous) for i in spring_range] #Maximum stretch of detailed serpentine
nonlin_force_det = xp.vars(3, spring_range, name="nonlin_force_det", lb=1e-6, ub=xp.infinity, vartype=xp.continuous)
nonlin_force_surr = xp.vars(3, spring_range, name="nonlin_force_surr", lb=1e-6, ub=xp.infinity, vartype=xp.continuous)

#New ones
z_array = xp.vars(spring_range, leg_range, name="z_array", vartype=xp.binary)
s3 = xp.vars(spring_range, leg_range, name="s3", lb=-xp.infinity, ub=xp.infinity, vartype=xp.continuous)
s4 = xp.vars(spring_range, leg_range, name="s4", lb=-xp.infinity, ub=xp.infinity, vartype=xp.continuous)
width = xp.vars(spring_range, name="width", lb=0.01, ub=0.03, vartype=xp.continuous)
p.addVariable (leg,angle,ntn_half,k0,stiffness,str_det,max_str_det,nonlin_force_det, nonlin_force_surr, z_array, s3, s4, width)

In [5]:
# Define design limits:
# Node-to-node distance limit of each leg topology
leg_ntn_half_limits = {2:(0.45, 0.8), 4:(0.70, 1.05), 6:(0.90, 1.25)} #UB: +-0.00

#Below has the following form:
# k = k0 * (c0 + c1*w + c2*w^2 + ...) (Obtained from FICO_solveFor_kAct_noint.ipynb)
k0_w_params = [0.07629573436862483, 0.6320653441409192, 1128.9394867101396, 803279.625862104, 121003.76568478544]


In [1]:
# Input polynomial coefficients obtained from curve fitting in nleg_Sum.summary Optimus files.
import pickle
import os
curpath = os.getcwd()
stiff_coeffs_quad = pickle.load(open(os.path.join(curpath,"integral_stiff_coeffs_quad_NT2_001_RF1_246v4.p"), "rb")) #To change after characterizing new serpentine variations for w=0.01. Gives k0 = f(ntn_half, angle)
max_str_coeffs_quad = pickle.load(open(os.path.join(curpath,"integral_max_str_coeffs_quad_NT2_001_RF1_246v4.p"), "rb")) #To change after characterizing new serpentine variations for w=0.01. Gives max_str_det = f(ntn_half, angle)

# Define objective stiffnesses as calculated by surrogate analysis of Optimus.
obj_stiffnesses = pickle.load(open(os.path.join(curpath,"obj_stiffnesses_FinalNom_2D_feasReg_246.p"), "rb")) #To change after LH300 random analysis after surr analysis. Not directly used in optimization

# Define required stretch amount of each serpentine as determined by surrogate analysis
req_str = pickle.load(open(os.path.join(curpath,"obj_req_str_FinalNom_2D_feasReg_246.p"), "rb")) #To change after LH300 random analysis after surr analysis

# Define f01, f02, ... f10 = f(dmax, k) 
FvsDmax_params = pickle.load (open(os.path.abspath(os.path.join(curpath, '..', "nonlinearStiffness", "summary&model_files", "246leg_v4", "LH_quadmodel_RF1_v4.p")), 'rb')) #To change after LH random analysis

# Define coefficients for f01, f02,... = f(dmax) for surrogate
surr_polycoeffs = pickle.load(open(os.path.join(curpath,"surr_polycoeffs_FinalNom_2D_feasReg_246.p"), "rb")) #To change after LH random analysis after surr analysis after getDmaxVsForceFromSurrogate in constraintsatisfaction/main.py

In [2]:
obj_stiffnesses

[6.154953e-05,
 5.453183e-05,
 1.760622e-05,
 2.035844e-05,
 4.894527e-05,
 5.205485e-05,
 4.411627e-05,
 1.361863e-05,
 1.162833e-06,
 2.868243e-05,
 7.94592e-06,
 6.906133e-06,
 7.974112e-06,
 1.623581e-05,
 1.655526e-05,
 9.210877e-05,
 9.068399e-05,
 7.805217e-05,
 2.10178e-05,
 1.08439e-06,
 4.041192e-06,
 4.163428e-05,
 4.196868e-05,
 6.678377e-05,
 7.254333e-05,
 3.042636e-05,
 5.848974e-05,
 5.474827e-05,
 3.948502e-05,
 8.006018e-05,
 8.17815e-05,
 4.121403e-05,
 1.811742e-05,
 3.505339e-06,
 1.389842e-06,
 1.257809e-05,
 7.485408e-05,
 8.276385e-05,
 8.324042e-05,
 1.301892e-05,
 1.106038e-05,
 1.691814e-05,
 1.915237e-05,
 1.600755e-05,
 1.442454e-05,
 1.148857e-06,
 1.678854e-05,
 2.397962e-05,
 5.053608e-05,
 4.748688e-05,
 1.382172e-05,
 1.027867e-05,
 6.838308e-05,
 6.769559e-05,
 6.290538e-06,
 1.469342e-05,
 2.43326e-05,
 2.30743e-05,
 2.828337e-05,
 7.800333e-05,
 7.598354e-05,
 2.499749e-05,
 2.743944e-06,
 7.545263e-05,
 5.415907e-05,
 5.315364e-05,
 4.039588e-06,
 

In [7]:
# a) Serpentine responses: These constraints describe how stiffness and max_lin_stretch variables of serpentines vary with changing leg, angle, and ntn_length.
# Define equations that describe serpentine stiffness and max. lin. stretch as functions of leg, angle, and ntn_length
for i in spring_range:
    k0[i] = 1e4 * (stiff_coeffs_quad[legs][0] + stiff_coeffs_quad[legs][1]*leg[i] +
                        stiff_coeffs_quad[legs][2]*angle[i] + stiff_coeffs_quad[legs][3]*ntn_half[i] + 
                        stiff_coeffs_quad[legs][4]*leg[i]*leg[i] + stiff_coeffs_quad[legs][5]*leg[i]*angle[i] + 
                        stiff_coeffs_quad[legs][6]*leg[i]*ntn_half[i] + stiff_coeffs_quad[legs][7]*angle[i]*angle[i] +
                        stiff_coeffs_quad[legs][8]*angle[i]*ntn_half[i] + 
                        stiff_coeffs_quad[legs][9]*ntn_half[i]*ntn_half[i])
    
    max_str_det[i] = max_str_coeffs_quad[legs][0] + max_str_coeffs_quad[legs][1]*leg[i] + \
                        max_str_coeffs_quad[legs][2]*angle[i] + max_str_coeffs_quad[legs][3]*ntn_half[i] + \
                        max_str_coeffs_quad[legs][4]*leg[i]*leg[i] + max_str_coeffs_quad[legs][5]*leg[i]*angle[i] + \
                        max_str_coeffs_quad[legs][6]*leg[i]*ntn_half[i] + max_str_coeffs_quad[legs][7]*angle[i]*angle[i] + \
                        max_str_coeffs_quad[legs][8]*angle[i]*ntn_half[i] + \
                        max_str_coeffs_quad[legs][9]*ntn_half[i]*ntn_half[i]
    
    p.addConstraint (stiffness[i] == k0[i] * (k0_w_params[0] + k0_w_params[1]*width[i] + k0_w_params[2]*width[i]**2
                        + k0_w_params[3]*width[i]**3 + k0_w_params[4]*width[i]**4))
    p.addConstraint (str_det[i] <= max_str_det[i])

In [8]:
# b) Design limitation constraints in local level: These define serpentine design constraints such as 8 leg serpentine cannot have ntn=0.35 (that'd be too tight)
for i in spring_range:
    p.addConstraint (xp.Sum(j*z_array[i,j] for j in leg_range) == leg[i])
    p.addConstraint (xp.Sum(z_array[i,j] for j in leg_range) == 1)

for i in spring_range:
    for j in leg_range:
        p.addConstraint (ntn_half[i] >= leg_ntn_half_limits[j][0] + s3[i,j])
        p.addConstraint (ntn_half[i] <= leg_ntn_half_limits[j][1] + s4[i,j])

        p.addIndicator (z_array[i,j] == 1, s3[i,j] >= 0)
        p.addIndicator (z_array[i,j] == 1, s3[i,j] <= 0)
        p.addIndicator (z_array[i,j] == 1, s4[i,j] >= 0)
        p.addIndicator (z_array[i,j] == 1, s4[i,j] <= 0)

In [9]:
# d) Global design limitations: Angle and distance restrictions between serpentines so that they won't clash when assembled.
    # i) The ntn distances of all horizontal springs at each column are same (0, 6, 12,...), (1, 8, 15,...)
first_vert_spr = (col_cnt-1)*row_cnt
for i in range(col_cnt-1):
    for j in range(1,row_cnt):
        p.addConstraint(ntn_half[i] == ntn_half[(col_cnt-1)*j+i])
        
    # ii) The ntn distances of all vertical springs at each row are same (54, 62, 70,...), (55, 63, 71,...)
for i in range(row_cnt-1):
    for j in range(1,col_cnt):
        p.addConstraint(ntn_half[i+first_vert_spr] == ntn_half[(row_cnt-1)*j+first_vert_spr+i])
    
    # iii) The ntn distances of horizontal springs should be greater than max(tan(angle_j)*dj), where j are the vertical springs in the connecting columns of horizontal spring i, so that vertical serpentines do not clash
for i in range(col_cnt-1):
    maxcandidates = list()
    for j in range(row_cnt-1):
        c = xp.tan(angle[first_vert_spr+((row_cnt-1)*i)+j]*math.pi/180) * ntn_half[first_vert_spr+((row_cnt-1)*i)+j] #Vertical spring on the left-side.
        d = xp.tan(angle[first_vert_spr+((row_cnt-1)*(i+1))+j]*math.pi/180) * ntn_half[first_vert_spr+((row_cnt-1)*(i+1))+j] #Vertical spring on the right-side.
        maxcandidates.append(c+d)
    p.addConstraint( (ntn_half[i]*2) >= xp.max(maxcandidates[0], maxcandidates[1], maxcandidates[2], maxcandidates[3], maxcandidates[4], maxcandidates[5], maxcandidates[6], maxcandidates[7]))
        
    # iv) The ntn distances of vertical springs should be greater than max(tan(angle_j)*dj), where j are the horizontal springs in the connecting columns of vertical spring i, so that vertical serpentines do not clash
for i in range(row_cnt-1):
    maxcandidates = list()
    for j in range(col_cnt-1):
        a = xp.tan(angle[((col_cnt-1)*i)+j]*math.pi/180) * ntn_half[((col_cnt-1)*i)+j] #Horizontal spring on the upper-side.
        b = xp.tan(angle[((col_cnt-1)*(i+1))+j]*math.pi/180) * ntn_half[((col_cnt-1)*(i+1))+j] #Horizontal spring on the lower-side.
        maxcandidates.append(a+b)
    p.addConstraint( (ntn_half[i+first_vert_spr]*2) >= xp.max(maxcandidates[0], maxcandidates[1], maxcandidates[2], maxcandidates[3], maxcandidates[4], maxcandidates[5]))

In [10]:
#Notes for this block: 
#Based on req_lin_str[i] and obj_dmax[i] determine the closest (and one smaller) FvsDmax_params parameters (this corresponds to j). 
#This will give the Force at that j*0.1*dmax[i]. Do the same (same way no NN) in surrogate. Then do variance.


for i in spring_range:
    F_init_ix = 0 #=0 if f01
    F_mid_ix = 3 #=3 is f04 #Chosen manually to give a good figure about the curve.
    F_term_ix = 7 #=7 is f08 #Chosen manually to give a good figure about the curve.
    displ_init = max_str_det[i]/10*(F_init_ix+1)
    displ_mid = max_str_det[i]/10*(F_mid_ix+1)
    displ_term = max_str_det[i]/10*(F_term_ix+1)
    
    #Surrogate
    p.addConstraint (nonlin_force_surr[0,i] == 1e3 * (surr_polycoeffs[i][0] + surr_polycoeffs[i][1]*displ_init + surr_polycoeffs[i][2]*displ_init**2 + \
                     surr_polycoeffs[i][3]*displ_init**3 + surr_polycoeffs[i][4]*displ_init**4 + surr_polycoeffs[i][5]*displ_init**5 + \
                     surr_polycoeffs[i][6]*displ_init**6))

    p.addConstraint (nonlin_force_surr[1,i] == 1e3 * (surr_polycoeffs[i][0] + surr_polycoeffs[i][1]*displ_mid + surr_polycoeffs[i][2]*displ_mid**2 + \
                     surr_polycoeffs[i][3]*displ_mid**3 + surr_polycoeffs[i][4]*displ_mid**4 + surr_polycoeffs[i][5]*displ_mid**5 + \
                     surr_polycoeffs[i][6]*displ_mid**6))
    
    p.addConstraint (nonlin_force_surr[2,i] == 1e3 * (surr_polycoeffs[i][0] + surr_polycoeffs[i][1]*displ_term + surr_polycoeffs[i][2]*displ_term**2 + \
                     surr_polycoeffs[i][3]*displ_term**3 + surr_polycoeffs[i][4]*displ_term**4 + surr_polycoeffs[i][5]*displ_term**5 + \
                     surr_polycoeffs[i][6]*displ_term**6))


    #Detailed
    p.addConstraint (nonlin_force_det[0,i] == 1e3 * (FvsDmax_params[legs][F_init_ix][0] + FvsDmax_params[legs][F_init_ix][1]*stiffness[i]/1e4 + \
                     FvsDmax_params[legs][F_init_ix][2] * max_str_det[i] + FvsDmax_params[legs][F_init_ix][3]*(stiffness[i]/1e4)**2 + \
                     FvsDmax_params[legs][F_init_ix][4] * max_str_det[i] * stiffness[i]/1e4 + FvsDmax_params[legs][F_init_ix][5]*(max_str_det[i])**2))

    p.addConstraint (nonlin_force_det[1,i] == 1e3 * (FvsDmax_params[legs][F_mid_ix][0] + FvsDmax_params[legs][F_mid_ix][1]*stiffness[i]/1e4 + \
                     FvsDmax_params[legs][F_mid_ix][2] * max_str_det[i] + FvsDmax_params[legs][F_mid_ix][3]*(stiffness[i]/1e4)**2 + \
                     FvsDmax_params[legs][F_mid_ix][4] * max_str_det[i] * stiffness[i]/1e4 + FvsDmax_params[legs][F_mid_ix][5]*(max_str_det[i])**2))
    
    p.addConstraint (nonlin_force_det[2,i] == 1e3 * (FvsDmax_params[legs][F_term_ix][0] + FvsDmax_params[legs][F_term_ix][1]*stiffness[i]/1e4 + \
                     FvsDmax_params[legs][F_term_ix][2] * max_str_det[i] + FvsDmax_params[legs][F_term_ix][3]*(stiffness[i]/1e4)**2 + \
                     FvsDmax_params[legs][F_term_ix][4] * max_str_det[i] * stiffness[i]/1e4 + FvsDmax_params[legs][F_term_ix][5]*(max_str_det[i])**2))
                     

In [11]:
# Add objective:
f01_ratios = [nonlin_force_det[0,i] / (nonlin_force_surr[0,i]) for i in spring_range]
f01_mean = xp.Sum(f01_ratios)/len(spring_range)
f01_variance = xp.Sum((nonlin_force_det[0,i]/nonlin_force_surr[0,i] - f01_mean)**2 for i in spring_range)
# f01_variance = xp.Sum(xp.abs(nonlin_force_det[0,i]/nonlin_force_surr[0,i] - f01_mean) for i in spring_range)

fmid_ratios = [nonlin_force_det[1,i] / (nonlin_force_surr[1,i]) for i in spring_range]
fmid_mean = xp.Sum(fmid_ratios)/len(spring_range)
fmid_variance = xp.Sum((nonlin_force_det[1,i]/nonlin_force_surr[1,i] - fmid_mean)**2 for i in spring_range)

fterm_ratios = [nonlin_force_det[2,i] / (nonlin_force_surr[2,i]) for i in spring_range]
fterm_mean = xp.Sum(fterm_ratios)/len(spring_range)
fterm_variance = xp.Sum((nonlin_force_det[2,i]/nonlin_force_surr[2,i] - fterm_mean)**2 for i in spring_range)
# fterm_variance = xp.Sum(xp.abs(nonlin_force_det[1,i]/nonlin_force_surr[1,i] - fterm_mean) for i in spring_range)

devFromNomNTN = xp.Sum((ntn_half[i]*2 - 1.95)**2 for i in spring_range) #Nominal NTN is 1.95 mm for 246 legs.

devFromReqStr = xp.Sum((str_det[i] - req_str[i])**2 for i in spring_range)

totSize = xp.Sum(ntn_half[i]*2 for i in spring_range)

p.setObjective(f01_variance + fmid_variance*2 + fterm_variance*2 + devFromReqStr*10 + devFromNomNTN/10, xp.minimize)

In [12]:
# p.controls.xslp_convergenceops = 6175 #6175=(bits 0-4, 11, 12), default=(bits 0-9, 11, 12)
p.controls.xslp_convergenceops = 6271
p.controls.xslp_solver = 0 # =0 slp, =1 knitro

p.controls.xslp_iterlimit = 50000
# p.controls.xslp_mipalgorithm = 1
p.controls.xslp_heurstrategy = 3 #-1(auto select), 0(no heur), 1(basic), 2(enhanced), 3(extensive), 4(no limits)
p.controls.xslp_maxtime = -200
p.nlpoptimize('g')



.34  102670.                    132    0      148
4557 O -1.039E+08            406.41  102694.                    266    0      148
4558 O -1.053E+08            406.30  102663.                     70    0      148
4559 O -1.039E+08            406.26  102652.                     65    0      148
4560 O -1.053E+08            406.34  102675.                     89    0      148
4561 O -1.039E+08            406.30  102664.                    118    0      148
4562 O -1.053E+08            406.37  102679.                     86    0      148
4563 O -1.039E+08            406.33  102667.                    179    0      148
4564 O -1.053E+08            406.38  102689.                    208    0      148
4565 O -1.039E+08            406.35  102678.                    190    0      148
4566 O -1.053E+08            406.31  102667.                     80    0      148
4567 O -1.039E+08            406.29  102659.                     41    0      148
4568 O -1.053E+08            406.35  102675.    

In [13]:
print (p.attributes.xslp_nlpstatus)

1


In [14]:
print (p.getSolution(f01_ratios))
print (p.getSolution(f01_mean))
print (p.getSolution(fterm_ratios))
print (p.getSolution(f01_variance))
print (p.getSolution(fmid_variance))
print (p.getSolution(fterm_variance))
print (p.getSolution(devFromNomNTN))
print (p.getSolution(totSize))
print (p.getSolution(devFromReqStr))

[1.1080395153818798, 1.0896611099325406, 1.129543911274379, 1.0498003327490542, 1.107357460941913, 1.1131258872932812, 1.1094159808851123, 1.115542082622336, 1.307469616253533, 1.1028332400915146, 1.0984224073438078, 1.123439900607134, 1.107321525174123, 1.0801920356025225, 1.0898257745329587, 1.0528427057029819, 1.0668049877420367, 1.0992101366247784, 1.1033895005371257, 1.4373364112193143, 1.093871945669913, 1.1099988447468998, 1.094682519711243, 1.1050118327561804, 1.0962161655925349, 1.1077119156061082, 1.1011815629345092, 1.1169078174348535, 1.1128497305241705, 1.0652663262422433, 1.178217562185302, 1.1038135171016112, 1.1054944680083842, 1.1117540727126918, 1.211796644115503, 1.1019433955951548, 1.1053313869376908, 0.9769183059849886, 1.08033563029505, 1.1244907056910596, 1.100397890829146, 1.105152544124202, 1.1160492559409398, 1.0213540031820711, 1.1043643899213547, 1.3081849803985273, 1.1110167336033385, 1.1043988131731035, 1.111679516714969, 1.1545636533800203, 1.111779512250

In [15]:
print (p.getObjVal())
print (p.getSolution(leg))
print (p.getSolution(ntn_half))
print (p.getSolution(angle))
print (p.getSolution(stiffness))
print (p.getSolution(width))
print (p.getSolution(max_str_det))
print (p.getSolution(str_det))

1.933426939723745
[6.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 4.0, 6.0, 4.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 4.0, 4.0, 4.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 4.0, 4.0, 6.0, 4.0, 6.0, 6.0, 6.0, 4.0, 4.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 4.0, 4.0, 6.0, 4.0, 4.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 6.0, 4.0, 6.0, 4.0, 6.0, 4.0, 6.0, 4.0, 4.0, 6.0, 6.0, 6.0, 6.0, 6.0, 4.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0]
[0.9000000125436279, 1.0179789903969534, 0.9684233815199568, 0.9960288218038744, 0.900000009923972, 0.9851802118566266, 0.9000000125436279, 1.0179789903969534, 0.9684233815199568, 0.9960288218038744, 0.900000009923972, 0.9851802118566266, 0.9000000125436279, 1.0179789903969534, 0.9684233815199568, 0.9960288218038744, 0.900000009923972, 0.9851802118566266, 0.9000000125436279, 1.0179789903969534, 0.9684233815199568, 0.99602882180

In [16]:
for i in range(5):
    print ("DetailedInit: ", str(p.getSolution(nonlin_force_det[0,i])), " ... SurrogateInit: ", p.getSolution(nonlin_force_surr[0,i]))
print()
for i in range(5):
    print ("DetailedTerm: ", str(p.getSolution(nonlin_force_det[1,i])), " ... SurrogateTerm: ", p.getSolution(nonlin_force_surr[1,i]))

DetailedInit:  0.08685090328728823  ... SurrogateInit:  0.07838249636553397
DetailedInit:  0.074492627431942  ... SurrogateInit:  0.06836311469035886
DetailedInit:  0.029881802671591205  ... SurrogateInit:  0.02645475078332973
DetailedInit:  0.027122983291638418  ... SurrogateInit:  0.02583632567596255
DetailedInit:  0.07052406810878127  ... SurrogateInit:  0.06368681351439474

DetailedTerm:  0.38280371845876054  ... SurrogateTerm:  0.36498854585842166
DetailedTerm:  0.3284386842036967  ... SurrogateTerm:  0.31709989474052697
DetailedTerm:  0.13147715712948785  ... SurrogateTerm:  0.122658951902267
DetailedTerm:  0.1194450584775971  ... SurrogateTerm:  0.11923900621661157
DetailedTerm:  0.31079752493999174  ... SurrogateTerm:  0.2962817179103393


In [17]:
print ("Obtained initial stiffnesses w/0 width effect:")
obtained_stiffnesses_nowidth = list()
for i in spring_range:
    spr_stiffness = p.getSolution(k0[i]/1e4)
    obtained_stiffnesses_nowidth.append(spr_stiffness)
    if i < 5: print (spr_stiffness)

print ()
print ("Obtained initial stiffnesses:")
obtained_stiffnesses = list()
for i in spring_range:
    spr_stiffness = p.getSolution(stiffness[i]/1e4)
    obtained_stiffnesses.append(spr_stiffness)
    if i < 5: print (spr_stiffness)

print ()
print ("Objective initial stiffnesses:")
for i in range(5):
    print (obj_stiffnesses[i])

Obtained initial stiffnesses w/0 width effect:
3.270197500455639e-06
3.2103415417772775e-06
2.131719157283826e-06
1.9514974596181507e-06
3.109183325240783e-06

Obtained initial stiffnesses:
6.837109748155821e-05
5.950211550761134e-05
1.9890040802541055e-05
2.1317606113860748e-05
5.430706335984874e-05

Objective initial stiffnesses:
6.154953e-05
5.453183e-05
1.760622e-05
2.035844e-05
4.894527e-05


In [27]:
print ("Obtained initial stiffnesses through calculation:")
obtained_stiffnesses_calc = list()
for i in spring_range:
    stifficoeffs = p.getSolution(stiff_coeffs_quad[246])
    k0i = (stifficoeffs[0] + stifficoeffs[1]*leg[i] + stifficoeffs[2]*angle[i] + stifficoeffs[3]*ntn_half[i] +
              stifficoeffs[4]*leg[i]*leg[i] + stifficoeffs[5]*leg[i]*angle[i] + stifficoeffs[6]*leg[i]*ntn_half[i] +
              stifficoeffs[7]*angle[i]*angle[i] + stifficoeffs[8]*angle[i]*ntn_half[i] +
              stifficoeffs[9]*ntn_half[i]*ntn_half[i])
    stiffi = k0i * (k0_w_params[0] + k0_w_params[1]*width[i] + k0_w_params[2]*width[i]**2 + k0_w_params[3]*width[i]**3 + k0_w_params[4]*width[i]**4)
    stiffival = xp.evaluate(stiffi, problem=p)
    obtained_stiffnesses_calc.append(stiffival)
    if i<5: print (stiffival)

print ()
print ("Objective initial stiffnesses:")
for i in range(5):
    print (obj_stiffnesses[i])

Obtained initial stiffnesses through calculation:
6.837111306933631e-05
5.950211475719877e-05
1.98892407472556e-05
2.131778293457493e-05
5.4307063893917196e-05

Objective initial stiffnesses:
6.154953e-05
5.453183e-05
1.760622e-05
2.035844e-05
4.894527e-05


In [28]:
print ("Obtained f01 of detailed design:")
obtained_f01_det = list()
for i in spring_range:
    f01 = p.getSolution(nonlin_force_det[0,i])
    obtained_f01_det.append(f01)
    if i < 5: print (f01)

print()
print ("Obtained f01 of surrogate design:")
obtained_f01_surr = list()
for i in spring_range:
    f01 = p.getSolution(nonlin_force_surr[0,i])
    obtained_f01_surr.append(f01)
    if i < 5: print (f01)

Obtained f01 of detailed design:
0.08685090328728823
0.074492627431942
0.029881802671591205
0.027122983291638418
0.07052406810878127

Obtained f01 of surrogate design:
0.07838249636553397
0.06836311469035886
0.02645475078332973
0.02583632567596255
0.06368681351439474


In [29]:
print ("Obtained fterm of detailed design:")
obtained_fterm_det = list()
for i in spring_range:
    fterm = p.getSolution(nonlin_force_det[1,i])
    obtained_fterm_det.append(fterm)
    if i < 5: print (fterm)

print()
print ("Obtained fterm of surrogate design:")
obtained_fterm_surr = list()
for i in spring_range:
    fterm = p.getSolution(nonlin_force_surr[1,i])
    obtained_fterm_surr.append(fterm)
    if i < 5: print (fterm)

Obtained fterm of detailed design:
0.38280371845876054
0.3284386842036967
0.13147715712948785
0.1194450584775971
0.31079752493999174

Obtained fterm of surrogate design:
0.36498854585842166
0.31709989474052697
0.122658951902267
0.11923900621661157
0.2962817179103393


In [33]:
# Calculate population variance of stiffness manually (through obtained values)
import statistics
ratios = [obtained_stiffnesses[i]/obj_stiffnesses[i] for i in spring_range]
print (ratios)
print (statistics.pvariance(ratios))

[1.1108305373178027, 1.0911446673917113, 1.1297167025370043, 1.0471139298424017, 1.1095467112521544, 1.115789815040696, 1.1113608698441144, 1.1129946672674231, 1.2528974327407059, 1.1031834594407257, 1.0904660108830164, 1.115625688330186, 1.1000864986844077, 1.076436466192986, 1.0867419977809656, 1.0518662800683682, 1.072775646136246, 1.1005994542804591, 1.1025647182579397, 1.4026866389697665, 1.0750713305321367, 1.111664142760467, 1.0955584198397852, 1.1078464249346547, 1.098692092156503, 1.1082477413564136, 1.1035516565038423, 1.1196406026785413, 1.1145895135738935, 1.0661932801972536, 1.1895548011692376, 1.105161940600549, 1.1040421517840744, 1.092419491895886, 1.1630230636508994, 1.0977294013066374, 1.1083987039526, 0.9721866562643254, 1.0837265095363033, 1.122724992649033, 1.0955963794195887, 1.1031451856814727, 1.1151810855884425, 1.016216391890199, 1.101831372314495, 1.2517508050831088, 1.1094613253781722, 1.104151585299321, 1.1142596588133316, 1.1588778301388702, 1.109335119959

In [34]:
# Calculate population variance of f01 manually
ratios = [obtained_f01_det[i]/obtained_f01_surr[i] for i in spring_range]
print (ratios)
print (statistics.pvariance(ratios))

[1.1080395153818798, 1.0896611099325406, 1.129543911274379, 1.0498003327490542, 1.107357460941913, 1.1131258872932812, 1.1094159808851123, 1.115542082622336, 1.307469616253533, 1.1028332400915146, 1.0984224073438078, 1.123439900607134, 1.107321525174123, 1.0801920356025225, 1.0898257745329587, 1.0528427057029819, 1.0668049877420367, 1.0992101366247784, 1.1033895005371257, 1.4373364112193143, 1.093871945669913, 1.1099988447468998, 1.094682519711243, 1.1050118327561804, 1.0962161655925349, 1.1077119156061082, 1.1011815629345092, 1.1169078174348535, 1.1128497305241705, 1.0652663262422433, 1.178217562185302, 1.1038135171016112, 1.1054944680083842, 1.1117540727126918, 1.211796644115503, 1.1019433955951548, 1.1053313869376908, 0.9769183059849886, 1.08033563029505, 1.1244907056910596, 1.100397890829146, 1.105152544124202, 1.1160492559409398, 1.0213540031820711, 1.1043643899213547, 1.3081849803985273, 1.1110167336033385, 1.1043988131731035, 1.111679516714969, 1.1545636533800203, 1.111779512250

In [35]:
# Calculate population variance of fterm manually
ratios = [obtained_fterm_det[i]/obtained_fterm_surr[i] for i in spring_range]
print (ratios)
print (statistics.pvariance(ratios))

[1.048810223779596, 1.0357577837496539, 1.0718920640561733, 1.001728060871383, 1.0489932592939981, 1.0532671147156605, 1.0502238757663422, 1.0565698102067662, 1.2431424936911935, 1.0442817590296827, 1.0515241149584584, 1.0721539074804172, 1.056957461781958, 1.0360584160998572, 1.0422339483619547, 1.0082189218623996, 0.9881171643414359, 1.0481395773405409, 1.0491033854594736, 1.4604158726187892, 1.0564317516717856, 1.0489765561644557, 1.039768010562718, 1.0440969968214984, 1.0409023072968053, 1.0480629966866892, 1.044696658933334, 1.057001289319942, 1.0535083650200334, 1.019851572997092, 1.0542389345389849, 1.0447763848113212, 1.0471531089711847, 1.0701378113810687, 1.261442670594942, 1.0488224483899622, 1.044904590399344, 0.9576003601043402, 1.0184027286335264, 1.0689186821830132, 1.0510186021017431, 1.0509702386165913, 1.0542615782222315, 0.9795067286301676, 1.0502799893318808, 1.2367212379945483, 1.0538171881037208, 1.0460699831112958, 1.0516720229967265, 1.0832403953951892, 1.056848

In [36]:
filename = "FICO_small_feasReg_2D_246_v4"

csvlist_all = list()
csvlist_leg = dict()
csvlist_all.append ("BridgeLoc, leg count, bounding angle, ntn length, wire_width, nStiffness")
for i in leg_range:
    csvlist_leg[i] = ["BridgeLoc, leg count, bounding angle, ntn length, wire_width, nStiffness"]

for i in spring_range:
    leg_sol = p.getSolution(leg[i]) 
    angle_sol = p.getSolution(angle[i]) 
    ntn_half_sol = p.getSolution(ntn_half[i])
    wirewidth_sol = p.getSolution(width[i])
    csvlist_all.append (str(i)+',' +str(leg_sol)+',' +str(angle_sol)+',' +str(ntn_half_sol)+',' +str(wirewidth_sol)+',' +str(obtained_stiffnesses_calc[i]))
    csvlist_leg[leg_sol].append (str(i)+',' +str(leg_sol)+',' +str(angle_sol)+',' +str(ntn_half_sol)+',' +str(wirewidth_sol)+',' +str(obtained_stiffnesses_calc[i]))

In [37]:
with open ("serpentine_backdesigns/"+filename+"_all.csv", "w") as fo:
    for line in csvlist_all:
        if isinstance(line, str) == False:
            line = str(line)
        fo.writelines(line+"\n")
for i in leg_range:
    with open ("serpentine_backdesigns/"+filename+"_serpentine_{}_NTN.csv".format(i), "w") as fo:
        for line in csvlist_leg[i]:
            if isinstance(line, str) == False:
                line = str(line)
            fo.writelines(line+"\n")